<a href="https://colab.research.google.com/github/kjmobile/lb2/blob/main/9_KNN_Hub_Carrier_Markets_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9. K-Nearest Neighbors: Which Markets Do the Big Network Airlines Lead?

In some markets the leading airline is one of the three big **network airlines** (American, Delta, United), which
fly passengers through their hubs. In others, Southwest or another airline leads. Can we tell which kind a market is
from its competition numbers?

**K-nearest neighbors (KNN)** answers in the simplest possible way: find the $k$ most similar markets we already
know, and take a **majority vote**. There is no equation to fit; the model *is* the data. That is why KNN is called
a **lazy** learner.

Slides: W3.1, K-nearest neighbors, distance metrics.

> **Data.** The 200 busiest U.S. domestic markets in 2019 (both directions combined), read straight from the course
> GitHub repository. Source: Sabre Market Intelligence, licensed to Embry-Riddle.

> **Figures.** The figures in this notebook are pictures made by its full version (`9_KNN_Hub_Carrier_Markets_all.ipynb`), so you can focus on the code that fits the model and produces the results.

## 1. Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

routes = pd.read_csv('https://raw.githubusercontent.com/kjmobile/lb2/main/data/sabre_routes_2019.csv')
routes['market'] = routes['airport_1'] + '-' + routes['airport_2']
routes.head()

,airport_1,airport_2,distance_mi,passengers,avg_fare,lcc_share,hhi,top_airline,market
0,JFK,LAX,2470.0,3115042.0,349.29,0.321,0.269,DL,JFK-LAX
1,LGA,ORD,731.0,2600397.0,174.47,0.086,0.293,AA,LGA-ORD
2,LAX,SFO,337.0,2289910.0,104.55,0.203,0.212,AS,LAX-SFO
3,LAX,ORD,1741.0,1853120.0,187.84,0.146,0.332,UA,LAX-ORD
4,LAS,LAX,236.0,1831441.0,73.20,0.438,0.180,WN,LAS-LAX


| Column | Meaning |
|---|---|
| `airport_1`, `airport_2` | the two airports of the market |
| `distance_mi` | distance in miles |
| `passengers` | passengers in 2019 |
| `avg_fare` | average base fare ($) |
| `lcc_share` | share of passengers on low-cost airlines (0 to 1) |
| `hhi` | market concentration, the Herfindahl-Hirschman Index: square each airline's share and add them up (1 = one airline carries everyone) |
| `top_airline` | the airline with the most passengers: AA American, DL Delta, UA United, WN Southwest, AS Alaska, B6 JetBlue, HA Hawaiian, NK Spirit |

In [2]:
# 1 = led by a big network airline (American, Delta or United), 0 = led by another airline
routes['hub_led'] = routes['top_airline'].isin(['AA', 'DL', 'UA']).astype(int)
routes['hub_led'].value_counts()

hub_led
1    118
0     82
Name: count, dtype: int64

**Figure 1**

![Figure 1](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/9_fig1.png)

Markets led by the big three sit mostly on the left (few low-cost passengers). Markets led by other airlines spread
to the right.

## 2. Split, and a First Try

Inputs: low-cost share, concentration and distance. We keep 25% of the markets (50) as the test set.

In [3]:
X = routes[['lcc_share', 'hhi', 'distance_mi']]
y = routes['hub_led']
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.25, random_state=320, stratify=y)

print('train:', train_X.shape, ' test:', test_X.shape)
print('baseline (always guess the larger group):', round(test_y.mean(), 2))

train: (150, 3)  test: (50, 3)
baseline (always guess the larger group): 0.58


In [4]:
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(train_X, train_y)
print('test accuracy:', knn_raw.score(test_X, test_y))

test accuracy: 0.66


Barely better than guessing. Look at the neighbors it found for one market: 10% low-cost share, concentration 0.45,
1,000 miles.

In [5]:
new_market = pd.DataFrame({'lcc_share': [0.10], 'hhi': [0.45], 'distance_mi': [1000]})
distances, rows = knn_raw.kneighbors(new_market)

print('prediction:', knn_raw.predict(new_market))
routes.loc[train_X.index[rows[0]], ['market', 'lcc_share', 'hhi', 'distance_mi', 'top_airline']]

prediction: [0]


,market,lcc_share,hhi,distance_mi,top_airline
128,EWR-TPA,0.304,0.518,999.0,UA
39,MCO-ORD,0.285,0.282,1006.0,UA
165,MCO-MDW,0.983,0.967,991.0,WN
134,PDX-PHX,0.389,0.291,1010.0,WN
160,DEN-PDX,0.590,0.322,989.0,WN


## 3. Why: Distance Depends on Units

KNN measures similarity with the **Euclidean distance**, the straight-line distance between two points:

$$d(i, j) = \sqrt{\sum_k (x_{ik} - x_{jk})^2}$$

Distance in miles runs from about 100 to 2,700, while the shares run from 0 to 1. A gap of 10 miles counts as much
as a gap of 10 in low-cost share, which is impossible. So the "nearest" markets above are simply markets of about
the same length, including one where nearly every passenger flies a low-cost airline.

The fix is **standardization**: rescale each input to mean 0 and standard deviation 1, so each input counts in its
own units of spread.

$$z = \frac{x - \text{mean}}{\text{standard deviation}}$$

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(train_X)                       # learn the mean and spread from the training markets only
train_scaled = scaler.transform(train_X)
test_scaled = scaler.transform(test_X)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train_scaled, train_y)
print('test accuracy:', knn.score(test_scaled, test_y))

test accuracy: 0.86


A new market must be scaled the same way before we ask about it:

In [7]:
new_scaled = scaler.transform(new_market)
distances, rows = knn.kneighbors(new_scaled)

print('prediction:', knn.predict(new_scaled))
routes.loc[train_X.index[rows[0]], ['market', 'lcc_share', 'hhi', 'distance_mi', 'top_airline']]

prediction: [1]


,market,lcc_share,hhi,distance_mi,top_airline
19,DFW-ORD,0.144,0.449,802.0,AA
20,LGA-MIA,0.041,0.495,1099.0,AA
7,LAX-SEA,0.033,0.400,955.0,AS
145,EWR-MIA,0.037,0.510,1088.0,UA
16,DFW-LAX,0.212,0.420,1232.0,AA


Now the neighbors are markets with little low-cost competition, and most are led by American or United. The
prediction flips to 1.

## 4. Choosing k

A small $k$ follows every odd market (it overfits); a large $k$ blurs real differences. Choose $k$ with
cross-validation on the training markets (notebook 7), not with the test set.

In [8]:
from sklearn.model_selection import cross_val_score

rows_k = []
for k in [1, 3, 5, 7, 9, 15, 25]:
    cv = cross_val_score(KNeighborsClassifier(n_neighbors=k), train_scaled, train_y, cv=5)
    rows_k.append({'k': k, 'cv accuracy': cv.mean()})

k_scores = pd.DataFrame(rows_k).round(3)
k_scores

,k,cv accuracy
0,1,0.753
1,3,0.807
2,5,0.813
3,7,0.793
4,9,0.800
5,15,0.773
6,25,0.767


**Figure 2**

![Figure 2](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/9_fig2.png)

Cross-validation picks $k = 5$, the model we already scored on the test set in section 3 (0.86).

## 5. Where Does It Go Wrong?

In [9]:
from sklearn.metrics import confusion_matrix

# rows: true group (0 other, 1 big three); columns: predicted group
confusion_matrix(test_y, knn.predict(test_scaled))

array([[15,  6],
       [ 1, 28]], dtype=int64)

### Your turn

A 700-mile market has a **60%** low-cost share and concentration **0.5**. Which kind of market is it, and who are its
five nearest neighbors?

1. In the cell below, fill in the blanks (`___`) and run it. Remember to scale the new market first.
2. Then finish this sentence:
   *KNN gives this market a ___ chance of being led by American, Delta or United, because its 5 nearest neighbors
   are led by ___.*

<details><summary>Show answer</summary>

```python
market_2 = pd.DataFrame({'lcc_share': [0.6], 'hhi': [0.5], 'distance_mi': [700]})
market_2_scaled = scaler.transform(market_2)

print('P(big three leads):', knn.predict_proba(market_2_scaled)[0, 1])
distances, rows = knn.kneighbors(market_2_scaled)
routes.loc[train_X.index[rows[0]], ['market', 'lcc_share', 'hhi', 'top_airline']]
```

*KNN gives this market a **0.0** chance of being led by American, Delta or United, because its 5 nearest neighbors are led by Southwest (WN) or JetBlue (B6).*
</details>

In [ ]:
market_2 = pd.DataFrame({'lcc_share': [___], 'hhi': [0.5], 'distance_mi': [700]})
market_2_scaled = scaler.___(market_2)

print('P(big three leads):', knn.predict_proba(market_2_scaled)[0, 1])
distances, rows = knn.kneighbors(market_2_scaled)
routes.loc[train_X.index[rows[0]], ['market', 'lcc_share', 'hhi', 'top_airline']]

## 6. Summary

| Step | What we found |
|---|---|
| The idea | Find the k most similar known markets and take a majority vote |
| Raw units | Distance in miles swamped the shares; accuracy 0.66 against a 0.58 baseline |
| Standardized | Each input counts equally; accuracy 0.86 |
| Choosing k | Cross-validation on the training markets picked k = 5 |
| New cases | Must be scaled with the same scaler before asking the model |

---
# Challenge

1. Drop `distance_mi` and use only `lcc_share` and `hhi` (both already between 0 and 1). Do you still need to
   standardize? Try it with and without.
2. Use `KNeighborsClassifier(n_neighbors=5, p=1)` (Manhattan distance: add up the absolute gaps instead of the
   straight line). Does the test accuracy change?
3. KNN can also predict a number: `KNeighborsRegressor` averages the neighbors' values. Predict `avg_fare` from
   `distance_mi` and `lcc_share` (standardized, k = 5) and compare its test R² with the linear regression of
   notebook 1.

In [11]:
# Your answer